# Entrenar el LoRA de Lía (SD 1.5)
GPU gratis: **Kaggle** (Settings → Accelerator → GPU T4/P100, Internet ON) o **Colab** (Runtime → T4).

1. Sube tus 15–30 imágenes de la cara a una carpeta `lia_dataset/` (Kaggle: como Dataset; Colab: panel de archivos).
2. Ejecuta las celdas en orden. Resultado: `lia_character.safetensors` para descargar.

Todas las imágenes deben ser de la misma cara. Imágenes de IA creadas con la persona de `persona.yaml`; la cuenta se publica siempre etiquetada como IA.

In [ ]:
!pip install -q diffusers==0.31.0 transformers accelerate peft bitsandbytes safetensors
!wget -q https://raw.githubusercontent.com/huggingface/diffusers/v0.31.0/examples/dreambooth/train_dreambooth_lora.py

In [ ]:
import os, glob
# Ajusta la ruta: Kaggle -> /kaggle/input/<tu-dataset>, Colab -> /content/lia_dataset
DATA = next((p for p in ['/kaggle/input/lia-dataset','/content/lia_dataset','lia_dataset'] if os.path.isdir(p)), None)
assert DATA, 'No encuentro la carpeta del dataset'
print(len(glob.glob(DATA+'/*')), 'imágenes en', DATA)

In [ ]:
!accelerate launch train_dreambooth_lora.py \
  --pretrained_model_name_or_path=stable-diffusion-v1-5/stable-diffusion-v1-5 \
  --instance_data_dir={DATA} \
  --instance_prompt='photo of lia_character woman' \
  --output_dir=lia_lora \
  --resolution=512 --center_crop \
  --train_batch_size=1 --gradient_accumulation_steps=4 \
  --gradient_checkpointing --use_8bit_adam --mixed_precision=fp16 \
  --learning_rate=1e-4 --lr_scheduler=constant --lr_warmup_steps=0 \
  --rank=16 --max_train_steps=1500 --checkpointing_steps=500 --seed=42

## Resultado
El script guarda `lia_lora/pytorch_lora_weights.safetensors`. Cópialo a `ComfyUI/models/loras/` como `lia_character.safetensors` y úsalo con `LORA=lia_character.safetensors python run.py 1`.

⚠️ Está en formato *diffusers*. ComfyUI suele cargarlo bien, pero si da error de claves, convierte con el script de kohya `convert_diffusers_lora_to_webui` (o entrena con kohya_ss).

In [ ]:
import shutil
shutil.copy('lia_lora/pytorch_lora_weights.safetensors','lia_character.safetensors')
try:
    from google.colab import files; files.download('lia_character.safetensors')
except ImportError:
    print('Kaggle: descárgalo desde la pestaña Output → lia_character.safetensors')